## RAG Day 3

### Expert Question Answerer for InsureLLM

LangChain 1.0 implementation of a RAG pipeline.

Using the VectorStore we created last time (with HuggingFace `all-MiniLM-L6-v2`)

In [5]:
%cd sample_data/llm_engineering/week5

/content/sample_data/llm_engineering/week5


In [12]:
!pip install langchain_ollama

In [3]:
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_ollama import ChatOllama

from langchain_chroma import Chroma
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_huggingface import HuggingFaceEmbeddings
import gradio as gr

In [6]:
import subprocess

# Start the command in a background process
process = subprocess.Popen(["ollama", "serve"])

# The kernel can continue execution while the process runs in the background
print("The 'ollama serve' process is running in the background.")

The 'ollama serve' process is running in the background.


In [7]:
!ollama list

NAME           ID              SIZE      MODIFIED          
llama3.2:1b    baf6a787fdff    1.3 GB    About an hour ago    


In [ ]:
MODEL = "gpt-4.1-nano"
DB_NAME = "vector_db"
load_dotenv(override=True)

In [9]:
from openai import OpenAI
DB_NAME = "vector_db"
OLLAMA_BASE_URL = "http://localhost:11434/v1"
ollama_client = OpenAI(base_url=OLLAMA_BASE_URL, api_key="ollama")
MODEL = 'llama3.2:1b'

### Connect to Chroma; use Hugging Face all-MiniLM-L6-v2

In [10]:
DB_NAME = "vector_db"
MODEL = 'llama3.2:1b'
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vectorstore = Chroma(persist_directory=DB_NAME, embedding_function=embeddings)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

### Set up the 2 key LangChain objects: retriever and llm

#### A sidebar on "temperature":
- Controls how diverse the output is
- A temperature of 0 means that the output should be predictable
- Higher temperature for more variety in answers

Some people describe temperature as being like 'creativity' but that's not quite right
- It actually controls which tokens get selected during inference
- temperature=0 means: always select the token with highest probability
- temperature=1 usually means: a token with 10% probability should be picked 10% of the time

Note: a temperature of 0 doesn't mean outputs will always be reproducible. You also need to set a random seed. We will do that in weeks 6-8. (Even then, it's not always reproducible.)

Note 2: if you want creativity, use the System Prompt!

In [12]:
import os
if os.path.exists(db_name):
    Chroma(persist_directory=db_name, embedding_function=embeddings).delete_collection()

vectorstore = Chroma.from_documents(documents=chunks, embedding=embeddings, persist_directory=db_name)
print(f"Vectorstore created with {vectorstore._collection.count()} documents")

NameError: name 'db_name' is not defined

In [8]:
retriever = vectorstore.as_retriever()
llm = ChatOllama(temperature=0, model=MODEL)

### These LangChain objects implement the method `invoke()`

In [9]:
retriever.invoke("Who is Avery?")

[]

In [20]:
llm.invoke("Who is Avery?")

AIMessage(content="I don't have any information on a specific person named Avery. Could you please provide more context or details about who Avery is or what you are referring to? This will help me provide a more accurate and helpful response.", additional_kwargs={}, response_metadata={'model': 'llama3.2:1b', 'created_at': '2026-10-01T09:03:38.832466639Z', 'done': True, 'done_reason': 'stop', 'total_duration': 25625703251, 'load_duration': 10994117719, 'prompt_eval_count': 29, 'prompt_eval_duration': 2078565000, 'eval_count': 45, 'eval_duration': 12545249000, 'logprobs': None, 'model_name': 'llama3.2:1b', 'model_provider': 'ollama'}, id='lc_run--01a0f6b4-192f-7230-b461-331d980642eb-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 29, 'output_tokens': 45, 'total_tokens': 74})

## Time to put this together!

In [21]:
SYSTEM_PROMPT_TEMPLATE = """
You are a knowledgeable, friendly assistant representing the company Insurellm.
You are chatting with a user about Insurellm.
If relevant, use the given context to answer any question.
If you don't know the answer, say so.
Context:
{context}
"""

In [22]:
def answer_question(question: str, history):
    docs = retriever.invoke(question)
    context = "\n\n".join(doc.page_content for doc in docs)
    system_prompt = SYSTEM_PROMPT_TEMPLATE.format(context=context)
    response = llm.invoke([SystemMessage(content=system_prompt), HumanMessage(content=question)])
    return response.content

In [ ]:
answer_question("Who is Averi Lancaster?", [])

'Avery Lancaster is the Co-Founder and Chief Executive Officer (CEO) of Insurellm.'

## What could possibly come next? 😂

In [24]:
gr.ChatInterface(answer_question).launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://a57717d926cf0aa96c.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


## Admit it - you thought RAG would be more complicated than that!!